In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
files = sorted(glob.glob('/home/ulyanov/data/sdo/hmi/nonpolar/*'))
#files = sorted(glob.glob('D:\\data\\sdo\\hmi\\nonpolar\\*'))
print(len(files))
print(files[0], files[-1])

357
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_000000_TAI.3.magnetogram.fits /home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241130_220000_TAI.3.magnetogram.fits


In [3]:
nx, ny = 1024, 1024
xc, yc = 511.5, 511.5
Rsun = 480


view_north = View(nx, ny, xc, yc, Rsun, -90, 90, 0, rsun_arc=-30 * 3600) ### North pole view
grid = view_north.grid()

mean, variance, coverage = np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0])

plt.ioff()

for file in files[:]:
    #print(file)

    with fits.open(file) as hdul:
        header = hdul[1].header.copy()
        data = hdul[1].data.copy()

    data = rebin(data, 8, update_header=header)
    view = View.from_header(header)

    transform = (view_north.to_carrington(correct_mu=True) -
                 view.to_synoptic(correct_mu=True, correct_dr=True, mu_thr=0.1, stonyhurst=False))
    grid_, alpha = transform(grid)
    data = bilinear(data, *grid_) * alpha

    n = (~np.isnan(data)) * (1 / alpha) ** 4
    coverage += np.nan_to_num(n)
    mean_ = mean.copy()
    mean += np.nan_to_num((data - mean) * n / coverage)
    variance += np.nan_to_num((data - mean) * (data - mean_) * n)


plt.ion()

variance = variance / coverage
mean[coverage == 0] = np.nan
coverage[coverage == 0] = np.nan
variance[coverage == 0] = np.nan



/tmp/ipykernel_62393/126925597.py:37: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage


In [5]:
show_data(mean, view_north, label=r'$B_{los}$')